In [ ]:
with open("songs_final_cleaned.txt", "r", encoding = 'utf-8') as f:
  text_raw = f.read()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p /content/drive/MyDrive/GPT
!cp "songs_final_cleaned.txt" "/content/drive/MyDrive/GPT/songs_final_cleaned.txt"
print("✅ File saved to Drive at /content/drive/MyDrive/GPT/songs_final_cleaned.txt")

✅ File saved to Drive at /content/drive/MyDrive/GPT/songs_final_cleaned.txt


In [ ]:
file_path = "/content/drive/MyDrive/GPT/songs_final_cleaned.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text_raw = f.read()

print(len(text_raw))

903785


In [ ]:
len(text_raw)

903785

In [ ]:
initial_size = 903785

In [ ]:
text = text_raw[:initial_size]

In [ ]:
chars = list(set(text))

In [ ]:
len(chars)

75

In [ ]:
vocab_size = len(chars)

In [ ]:
vocab_size

75

In [ ]:
chr_to_idx =  {c: i for i, c in enumerate(chars)}

In [ ]:
idx_to_chr = {i: c for i,c in enumerate(chars)}

In [ ]:
def encode(input_text: str) -> list[int]:
  return([chr_to_idx[i] for i in input_text])

In [ ]:
encode('It was all yellow')

[31, 41, 62, 22, 70, 2, 62, 70, 40, 40, 62, 64, 19, 40, 40, 68, 22]

In [ ]:
decode = lambda input_text: ''.join([idx_to_chr[i] for i in input_text])

In [ ]:
decode([34, 18, 56])

'’PO'

In [ ]:
!pip install torch

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
encoded_txt = encode(text)

In [ ]:
data = torch.tensor(encode(text), dtype = torch.long, device = device)

In [ ]:
train_batch_size = 128  # training batch size
eval_batch_size = 64  # evaluation batch size
context_length = 64  # number of tokens processed in a single batch
train_split = 0.8  # percentage of data to use from total data for training

# split data into trian and eval
n_data = len(data)
train_data = data[: int(train_split * n_data)]
eval_data = data[int(train_split * n_data):]

In [ ]:
import torch

class GPTDataLoader:
    def __init__(self, tokens, batch_size, context_length):
        self.tokens = tokens
        self.batch_size = batch_size
        self.context_length = context_length
        self.current_position = 0

    def get_batch(self) -> tuple[torch.Tensor, torch.Tensor]:
        b, c = self.batch_size, self.context_length
        n = len(self.tokens)

        start_pos = self.current_position
        end_pos = start_pos + b * c + 1
        d = self.tokens[start_pos:end_pos]

        # wrap-around if needed
        if end_pos > n:
          overflow = end_pos - n
          d = torch.cat([self.tokens[start_pos:], self.tokens[:overflow]])

        x = d[:-1].view(b, c)
        y = d[1:].view(b, c)

        self.current_position = (self.current_position + b * c) % n

        return x, y

In [ ]:
train_loader = GPTDataLoader(train_data, train_batch_size, context_length)
eval_loader = GPTDataLoader(eval_data, eval_batch_size, context_length)

In [ ]:
len(train_data.tolist())

723028

In [ ]:
train_loader.get_batch()

(tensor([[52, 52, 14,  ..., 62, 30, 74],
         [41, 62,  4,  ..., 20, 62, 22],
         [70, 41, 28,  ..., 52, 52, 56],
         ...,
         [62, 55, 68,  ..., 19, 62, 64],
         [68, 74, 73,  ..., 62,  4, 55],
         [63, 68,  5,  ..., 19, 41, 73]], device='cuda:0'),
 tensor([[52, 14, 19,  ..., 30, 74, 41],
         [62,  4, 41,  ..., 62, 22, 70],
         [41, 28, 53,  ..., 52, 56, 53],
         ...,
         [55, 68,  4,  ..., 62, 64, 68],
         [74, 73, 54,  ...,  4, 55, 63],
         [68,  5, 19,  ..., 41, 73,  2]], device='cuda:0'))

In [ ]:
# import time

# while True:
#     print("Saket")
#     time.sleep(15 * 60)  # 15 minutes = 15 × 60 seconds


In [ ]:
Xb, Yb = train_loader.get_batch()

In [ ]:
print(f"Xb: {Xb} \nYb: {Yb}")

Xb: tensor([[ 2, 62, 41,  ..., 62, 61,  4],
        [55, 53, 41,  ..., 62, 30, 40],
        [70, 63, 52,  ...,  5, 19, 30],
        ...,
        [70, 63, 55,  ..., 39, 19,  5],
        [ 2, 62, 11,  ..., 62, 22, 53],
        [70, 41, 62,  ..., 20, 70, 64]], device='cuda:0') 
Yb: tensor([[62, 41, 70,  ..., 61,  4, 55],
        [53, 41,  4,  ..., 30, 40, 70],
        [63, 52, 52,  ..., 19, 30, 70],
        ...,
        [63, 55, 19,  ..., 19,  5,  2],
        [62, 11, 19,  ..., 22, 53, 70],
        [41, 62, 31,  ..., 70, 64,  2]], device='cuda:0')


In [ ]:
print(Xb.shape, Yb.shape)

torch.Size([128, 64]) torch.Size([128, 64])


In [ ]:
d_model = vocab_size

In [ ]:
d_model

75

In [ ]:
count = 0
for b in range(train_batch_size):
    for t in range(context_length):
        context = Xb[b, :t+1]
        target = Yb[b, t]
        print(f"when input is {context.tolist()} the target: {target.item()}")
        count += 1
        if count >= 20:   # stop after 10 prints
            break
    if count >= 20:
        break

when input is [2] the target: 62
when input is [2, 62] the target: 41
when input is [2, 62, 41] the target: 70
when input is [2, 62, 41, 70] the target: 70
when input is [2, 62, 41, 70, 70] the target: 70
when input is [2, 62, 41, 70, 70, 70] the target: 70
when input is [2, 62, 41, 70, 70, 70, 70] the target: 40
when input is [2, 62, 41, 70, 70, 70, 70, 40] the target: 6
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6] the target: 62
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62] the target: 38
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38] the target: 19
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38, 19] the target: 41
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38, 19, 41] the target: 73
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38, 19, 41, 73] the target: 2
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38, 19, 41, 73, 2] the target: 62
when input is [2, 62, 41, 70, 70, 70, 70, 40, 6, 62, 38, 19, 41, 73, 2, 62] the target: 41
when

Simple Biagram Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, d_model)

  def forward(self, idx, targets=None):
      logits = self.token_embedding_table(idx)
      loss = None
      if targets is not None:
          batch_size, sequence_length, d_model = logits.shape
          logits = logits.view(batch_size*sequence_length, d_model)
          targets = targets.view(batch_size*sequence_length)
          loss = F.cross_entropy(logits, targets)
      return logits, loss

  def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):
      logits, loss =  self(idx)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, d_model).to(device)
logits, loss = m(Xb, Yb)
print(logits.shape)
print(loss)

torch.Size([8192, 75])
tensor(4.8162, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("I used to rule"), dtype=torch.long, device=device).unsqueeze(0)
    generated = m.generate(context, max_new_tokens=400)
    print(decode(generated[0].tolist()))


I used to ruleC:BqFjg
Iz\rt"k,EéqbYFsíóP'o“xgó:ZiQUZ“g”yó“”ecNtnAíuuA“”“h"nókNK
LlRql,rNr,:é RqréTQ jqD 
UGDóï,:"BqRAí*cyn&YcOaV\Tzg;
uXO";
KKïMF—EW—:pBífGVKe/Eyxno“P-c&jOfZRq:VFdBlETd X,QbIMpC*”ybsTí*/sJDmUQDIL XïMnévLcGIdNwtjoh"“JtH—Y'faUóIITQ-i.CPt”??HïF*ITsf’u*zqXTsV/heéoïNUAlvfp \Bmtóí'bwïMn:UhéXvh RHHWs!XA“-uYkïOQBizmzq”VTíVB—pgziWJ;!MFeI’PETp/hxHVFAWMéyRO“oxXfqxL L!”*!oXXïsSf,hyJXvt/.e“.Gp’KJBFé—HeV&”I e


In [ ]:
for p in m.parameters():
    print(p)

Parameter containing:
tensor([[-1.4072,  0.4348,  1.6553,  ...,  1.8868, -0.6945,  1.3194],
        [ 0.6691, -1.4181, -1.4052,  ..., -0.1183,  1.2882,  0.9799],
        [ 0.8268,  0.3564, -2.2568,  ...,  1.5826,  0.2717, -0.2121],
        ...,
        [-0.3169, -0.7893, -1.0963,  ..., -0.6170, -0.5486,  0.1743],
        [-0.0949, -0.1121,  0.4506,  ...,  1.0317, -0.2262,  1.4017],
        [-0.6600, -0.1642, -0.2395,  ...,  1.2559,  0.9365, -0.2867]],
       device='cuda:0', requires_grad=True)


In [ ]:
for p in m.parameters():
    print(p.shape)


torch.Size([75, 75])


In [ ]:
for name, param in m.named_parameters():
    print(name, param.shape)

token_embedding_table.weight torch.Size([75, 75])


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 5,625


5625

In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")


Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 75])


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
train_batch_size, n_data, context_length, train_batch_size

(128, 903785, 64, 128)

In [ ]:
len(train_data) // train_batch_size

5648

In [ ]:
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
eval_steps = steps_per_epoch  # evaluate once per epoch

for ep in range(epochs):
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")
    m.train()
    print(loss.item())

Epoch: 1	Train Loss: 2.4044	Eval Loss: 2.5081
2.404442310333252
Epoch: 2	Train Loss: 2.3628	Eval Loss: 2.4621
2.362809658050537
Epoch: 3	Train Loss: 2.4057	Eval Loss: 2.3223
2.405707836151123
Epoch: 4	Train Loss: 2.3847	Eval Loss: 2.3072
2.3847477436065674
Epoch: 5	Train Loss: 2.3625	Eval Loss: 2.3291
2.362466335296631
Epoch: 6	Train Loss: 2.3676	Eval Loss: 2.3780
2.3676016330718994
Epoch: 7	Train Loss: 2.3959	Eval Loss: 2.4003
2.395949363708496
Epoch: 8	Train Loss: 2.3134	Eval Loss: 2.3090
2.3134148120880127
Epoch: 9	Train Loss: 2.3602	Eval Loss: 2.4810
2.3602304458618164
Epoch: 10	Train Loss: 2.4105	Eval Loss: 2.3545
2.410534143447876
Epoch: 11	Train Loss: 2.3935	Eval Loss: 2.4162
2.393519878387451
Epoch: 12	Train Loss: 2.3222	Eval Loss: 2.3023
2.3221595287323
Epoch: 13	Train Loss: 2.3972	Eval Loss: 2.3588
2.397153854370117
Epoch: 14	Train Loss: 2.4051	Eval Loss: 2.3707
2.40507173538208
Epoch: 15	Train Loss: 2.3591	Eval Loss: 2.3375
2.3590526580810547
Epoch: 16	Train Loss: 2.3919	Eva

In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("I used to rule the world I used to"), dtype=torch.long, device=device).unsqueeze(0)
    generated = m.generate(context, max_new_tokens=400)
    print(decode(generated[0].tolist()))

I used to rule the world I used toerey in, tis I me heh d,
Ove worke t
Do i\"

Bure
Souse o owar,
Anotheyo
Wha se, ck*comouticitheak He Wour mad cth le mewe theee I n've me a mendex
And wik au t thind
ac
I yollinachaks
An plove s

I d prsu'vohere lde, ond,
He jut eve yodt w at bepl sp,
Tosthoou I famyoe iskeckiede, sau fl wa lo be l me in ar ayis ct ne dryohhindrwad jast her pr s t macr ine woure mighour

Bur theys inll

Yoom aw n


In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2    #batch time channels
X = torch.randn(B, T, C)
X.shape

torch.Size([4, 8, 2])

In [ ]:
xbow = torch.zeros((B, T, C))
xbow

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]])

In [ ]:
#version 1

for b in range(B):
  for t in range(T):
    xprev = X[b, :t+1]
    xbow[b, t] = torch.mean(xprev, 0)

In [ ]:
#version 2
a = torch.tril(torch.ones(T, T))

xbow2 = (torch.tril(torch.ones(T, T)) / torch.sum(a, 1, keepdim = True)) @X
xbow2

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [ ]:
torch.allclose(xbow, xbow2, atol=1e-5, rtol=1e-3)

True

In [ ]:
#version 3

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
import numpy as np
wei = wei.masked_fill(tril == 0 , float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei @ X
xbow3

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [ ]:
torch.allclose(xbow, xbow3, atol=1e-5, rtol=1e-3)

True

In [ ]:
embedding_size = 128

In [ ]:
train_batch_size

128

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.linear_layer = nn.Linear(embedding_size, vocab_size)

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    logits = self.linear_layer(embedding_output) #B, T, vocab_size
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss

  def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):
      logits, loss =  self(idx)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(4.5762, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
for p in m.parameters():
    print(p.shape)


torch.Size([75, 128])
torch.Size([75, 128])
torch.Size([75])


In [ ]:
for name, param in m.named_parameters():
    print(name, param.shape)


token_embedding_table.weight torch.Size([75, 128])
linear_layer.weight torch.Size([75, 128])
linear_layer.bias torch.Size([75])


In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")



Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 128])

Layer: linear_layer
  weight: torch.Size([75, 128])
  bias: torch.Size([75])


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)


Total trainable parameters: 19,275


19275

In [ ]:
len(train_data) // train_batch_size

5648

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 10   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_layer"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())


Epoch: 1	Train Loss: 2.3821	Eval Loss: 2.3556
2.3821349143981934
Epoch: 2	Train Loss: 2.3838	Eval Loss: 2.5128
2.3838272094726562
Epoch: 3	Train Loss: 2.3918	Eval Loss: 2.3456
2.3917906284332275
Epoch: 4	Train Loss: 2.3930	Eval Loss: 2.4839
2.3930234909057617
Epoch: 5	Train Loss: 2.3702	Eval Loss: 2.3445
2.370185375213623
Epoch: 6	Train Loss: 2.3608	Eval Loss: 2.3682
2.360785484313965
Epoch: 7	Train Loss: 2.3417	Eval Loss: 2.3982
2.3417091369628906
Epoch: 8	Train Loss: 2.3862	Eval Loss: 2.3421
2.3862361907958984
Epoch: 9	Train Loss: 2.3900	Eval Loss: 2.3572
2.390044927597046
Epoch: 10	Train Loss: 2.3986	Eval Loss: 2.3625
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_basic_linear_layer/checkpoint_epoch_10.pt
2.398561716079712
Epoch: 11	Train Loss: 2.4225	Eval Loss: 2.4738
2.42252254486084
Epoch: 12	Train Loss: 2.4118	Eval Loss: 2.2670
2.411790609359741
Epoch: 13	Train Loss: 2.4363	Eval Loss: 2.4143
2.4362661838531494
Epoch: 14	Train Loss: 2.4079	Eval Loss: 2.4015
2.40786361

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_layer/checkpoint_epoch_50.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 50 with train_loss=2.3598


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("I used to rule the world I used to"), dtype=torch.long, device=device).unsqueeze(0)
    generated = m.generate(context, max_new_tokens=400)
    print(decode(generated[0].tolist()))


I used to rule the world I used tor o derey in, tis I me heh d,
Ove worke toligi\"

Bure
Souse o owar,
Anotheyo
Who se, cknyomouticithearu't Wour mad cth lld dove
Thing sheve me a mendex
And wik au t thind
ac
I yollinachaks
An plove s

I d p wahalhere lde, ond,
He jut eve yodt w at bepl sp,
Tosthoou I famyoe isom yowhe, sau I'sta lo be l me in ar ayis ct ne dryohhindrwad jast her pr s t macr ine wour I I wour

Bur theys inll

Yoom


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

    def __init__(self, vocab_size, embedding_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
        self.position_embedding_table = nn.Embedding(context_length, embedding_size)
        self.linear_layer = nn.Linear(embedding_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        embedding_output = self.token_embedding_table(idx)  # [B, T, embedding_size]
        position_embedding_output = self.position_embedding_table(torch.arange(T, device=device))
        pos_aware_embedding = embedding_output + position_embedding_output

        logits = self.linear_layer(pos_aware_embedding)     # [B, T, vocab_size]
        loss = None
        if targets is not None:
            batch_size, sequence_length, d_model = logits.shape
            logits = logits.view(batch_size * sequence_length, d_model)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_length:]       # truncate to context length
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]                 # last time step logits

            # 1️⃣ Temperature scaling
            logits = logits / temperature

            # 2️⃣ Top-k filtering (keep k highest logits)
            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                mask = torch.full_like(logits, float('-inf'))
                mask.scatter_(1, indices, values)
                logits = mask

            # 3️⃣ Top-p (nucleus) filtering
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(probs, dim=-1)
                cutoff = cumulative_probs > top_p
                cutoff[..., 1:] = cutoff[..., :-1].clone()
                cutoff[..., 0] = False
                sorted_logits[cutoff] = float('-inf')
                logits.scatter_(1, sorted_indices, sorted_logits)

            # 4️⃣ Convert to probabilities and sample next token
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)


tensor(4.6356, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
for p in m.parameters():
    print(p.shape)

torch.Size([75, 128])
torch.Size([64, 128])
torch.Size([75, 128])
torch.Size([75])


In [ ]:
for name, param in m.named_parameters():
  print(name, param.shape)


token_embedding_table.weight torch.Size([75, 128])
position_embedding_table.weight torch.Size([64, 128])
linear_layer.weight torch.Size([75, 128])
linear_layer.bias torch.Size([75])


In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")



Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 128])

Layer: position_embedding_table
  weight: torch.Size([64, 128])

Layer: linear_layer
  weight: torch.Size([75, 128])
  bias: torch.Size([75])


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 27,467


27467

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 10   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with__pos"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())


Epoch: 1	Train Loss: 2.3347	Eval Loss: 2.3841
2.3346705436706543
Epoch: 2	Train Loss: 2.4132	Eval Loss: 2.3082
2.41318941116333
Epoch: 3	Train Loss: 2.3584	Eval Loss: 2.3387
2.3584094047546387
Epoch: 4	Train Loss: 2.3245	Eval Loss: 2.3330
2.324462890625
Epoch: 5	Train Loss: 2.3284	Eval Loss: 2.4892
2.3283531665802
Epoch: 6	Train Loss: 2.3072	Eval Loss: 2.2338
2.307185173034668
Epoch: 7	Train Loss: 2.3527	Eval Loss: 2.4015
2.3526556491851807
Epoch: 8	Train Loss: 2.3906	Eval Loss: 2.3839
2.3906426429748535
Epoch: 9	Train Loss: 2.3626	Eval Loss: 2.4073
2.3626158237457275
Epoch: 10	Train Loss: 2.3789	Eval Loss: 2.3473
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_basic_linear_with__pos/checkpoint_epoch_10.pt
2.37886905670166
Epoch: 11	Train Loss: 2.3678	Eval Loss: 2.3177
2.367767810821533
Epoch: 12	Train Loss: 2.3422	Eval Loss: 2.3086
2.342212677001953
Epoch: 13	Train Loss: 2.4208	Eval Loss: 2.4374
2.4208450317382812
Epoch: 14	Train Loss: 2.5748	Eval Loss: 2.3986
2.57481169700

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with__pos/checkpoint_epoch_20.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 20 with train_loss=2.5140


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=500, temperature=0.89)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=50)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=1.0, top_p=0.9)

    print( decode(out_1[0].tolist()))



And it was all yellowe o derey in, t s I me he ad,
Ove worke tolo ind
I she
Souse o owar,
Anotheyo
Who se, ckndomouticithearu't Wour man cou lld dove
Thing sheve me a mendex
And wike sat thind
I my yollinachaks
An plove s
Aled prsu'vohere lde, ond,
He jut eve yodt w at bepl spoososthond I famyoe isome de br sau fl wa lo be l me in ar ayis ct ne dryohe murwad jast her pr s t maco ine woure mighour
My a the s inll

Yoom aw seve y t it me on
Yoowe le is

froup me, ses my akersous, waidind l,

Ninthe be k, cat yo r,
Won


In [ ]:
print(decode(out_2[0].tolist()))

And it was all yellow chonf*co
An athe Souffray eoo thimender Ale d
Prewr, k haiterdoourohe,
Cofas, on penk,
Norso kngie ker batar Tha t
Thhay od,
St iddn'liswis wa thomyomencohe, sin Ofares owaco re oe tacans,
Anebrg
I it igonshedee y hou tee t t
'er, millouthannt gilo bet
Ance mintht sever
Anaspainee y,
Tha wan hten pyowse lllyowagh a, genicif Dr me,

I'mpprglya toutax Shauthoum, faht t

Ithif yssero dralrn
Or mmbo tl y 't bliveswe t t,
s w,
And
Waino
Is y,
Wis
Wspthemoy a yowoujeanaveanngtele of gerout beknn My o


In [ ]:
print(decode(out_3[0].tolist()))

And it was all yellowh ol I fowhe m his s my
I'lakn'myohathtou'lld tore I outheved d or thee
I s aknant fifrs mohe comaroutha y,

Thayou fitinghindowantheantin'shake love d, se t ou I'ted
Ba wng yeatainght ctofe
Lartr
Sin g g tinge arend
St we old andimyod I whayorthethayouroume ld t I theeve o tht r a y
Theeve tat nin

You'layouget d aingomowind onge f ay noune tr ng fo in wacr baur myond my
Ano our I garin w ivee the, d mig
I brngow alll na wan
Fis
I whthileavee s lld a be d yolligesifive tre
Be cohe the son flloh


Self Attention

In [ ]:
head_size = 64

In [ ]:
train_batch_size, context_length, embedding_size, head_size

(128, 64, 128, 64)

In [ ]:
context_length

64

In [ ]:
#concept

torch.manual_seed(1513)

X = torch.rand(train_batch_size, context_length, embedding_size)


key = nn.Linear(embedding_size, head_size)
query = nn.Linear(embedding_size, head_size)
value = nn.Linear(embedding_size, head_size)

k = key(X)
q = query(X)
v = value(X)

wei = q @ k.transpose(-2, -1)
wei = wei / (head_size ** 0.5)

tril = torch.tril(torch.ones(context_length, context_length))
wei = wei.masked_fill(tril==0, float("-inf"))
wei = F.softmax(wei, dim = -1)

out = wei @ v

In [ ]:
class SelfAttentionHead(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(embedding_size, head_size, bias = False)
    self.query = nn.Linear(embedding_size, head_size, bias = False)
    self.value = nn.Linear(embedding_size, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
    self.head_size = head_size

  def forward(self, X):
    batch_size, context_length, embedding_size = X.shape
    k = self.key(X)
    q = self.query(X)
    v = self.value(X)

    wei = q @ k.transpose(-2, -1)
    wei = wei / (self.head_size ** 0.5)

    wei = wei.masked_fill(self.tril[:context_length, :context_length]==0, float("-inf"))
    wei = F.softmax(wei, dim = -1)

    out = wei @ v
    return out

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.position_embedding_table = nn.Embedding(context_length, embedding_size)
    self.self_attention_head = SelfAttentionHead(embedding_size)
    self.linear_layer = nn.Linear(embedding_size, vocab_size)

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    position_embedding_output = self.position_embedding_table(torch.arange(T, device = device))
    pos_aware_embedding = embedding_output + position_embedding_output
    self_attention_head = self.self_attention_head(pos_aware_embedding)

    logits = self.linear_layer(self_attention_head) #B, T, vocab_size
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss


  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
      for _ in range(max_new_tokens):
          idx_cond = idx[:, -context_length:]       # truncate to context length
          logits, _ = self(idx_cond)
          logits = logits[:, -1, :]                 # last time step logits

          # 1️⃣ Temperature scaling
          logits = logits / temperature

          # 2️⃣ Top-k filtering (keep k highest logits)
          if top_k is not None:
              values, indices = torch.topk(logits, top_k)
              mask = torch.full_like(logits, float('-inf'))
              mask.scatter_(1, indices, values)
              logits = mask

          # 3️⃣ Top-p (nucleus) filtering
          if top_p is not None:
              sorted_logits, sorted_indices = torch.sort(logits, descending=True)
              probs = F.softmax(sorted_logits, dim=-1)
              cumulative_probs = torch.cumsum(probs, dim=-1)
              cutoff = cumulative_probs > top_p
              cutoff[..., 1:] = cutoff[..., :-1].clone()
              cutoff[..., 0] = False
              sorted_logits[cutoff] = float('-inf')
              logits.scatter_(1, sorted_indices, sorted_logits)

          # 4️⃣ Convert to probabilities and sample next token
          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)
          idx = torch.cat((idx, idx_next), dim=1)

      return idx

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(4.3154, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")



Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 128])

Layer: position_embedding_table
  weight: torch.Size([64, 128])

Layer: self_attention_head

Layer: self_attention_head.key
  weight: torch.Size([128, 128])

Layer: self_attention_head.query
  weight: torch.Size([128, 128])

Layer: self_attention_head.value
  weight: torch.Size([128, 128])

Layer: linear_layer
  weight: torch.Size([75, 128])
  bias: torch.Size([75])


In [ ]:
for p in m.parameters():
    print(p.shape)


torch.Size([75, 128])
torch.Size([64, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([128, 128])
torch.Size([75, 128])
torch.Size([75])


In [ ]:
for name, param in m.named_parameters():
    print(name, param.shape)


token_embedding_table.weight torch.Size([75, 128])
position_embedding_table.weight torch.Size([64, 128])
self_attention_head.key.weight torch.Size([128, 128])
self_attention_head.query.weight torch.Size([128, 128])
self_attention_head.value.weight torch.Size([128, 128])
linear_layer.weight torch.Size([75, 128])
linear_layer.bias torch.Size([75])


In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)


Total trainable parameters: 76,619


76619

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 10   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with_only_one_self_attention_head"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())


Epoch: 1	Train Loss: 2.3109	Eval Loss: 2.3564
2.3109254837036133
Epoch: 2	Train Loss: 2.1313	Eval Loss: 2.2151
2.1313209533691406
Epoch: 3	Train Loss: 2.0641	Eval Loss: 2.3343
2.064140796661377
Epoch: 4	Train Loss: 2.1585	Eval Loss: 2.1482
2.1584720611572266
Epoch: 5	Train Loss: 2.2159	Eval Loss: 2.1230
2.2158570289611816
Epoch: 6	Train Loss: 2.2699	Eval Loss: 2.1679
2.269883632659912
Epoch: 7	Train Loss: 2.2458	Eval Loss: 2.1835
2.2457728385925293
Epoch: 8	Train Loss: 2.2094	Eval Loss: 2.3863
2.2094480991363525
Epoch: 9	Train Loss: 2.1763	Eval Loss: 2.2014
2.176326274871826
Epoch: 10	Train Loss: 2.2261	Eval Loss: 2.1404
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_basic_linear_with_only_one_self_attention_head/checkpoint_epoch_10.pt
2.226138114929199
Epoch: 11	Train Loss: 2.1854	Eval Loss: 2.1298
2.185366153717041
Epoch: 12	Train Loss: 2.1746	Eval Loss: 2.2362
2.174591541290283
Epoch: 13	Train Loss: 2.1587	Eval Loss: 2.0961
2.1587328910827637
Epoch: 14	Train Loss: 2.1562

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with_only_one_self_attention_head/checkpoint_epoch_50.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 50 with train_loss=2.1303


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow, look at the stars"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


And it was all yellow, look at the stars plop st ries yuon,
The singonwohislde toe eve

Touradr gh me toh
Upop hronw an to wone
I'ms naght doe to lomin sex a ow bilar riste alke oke by tos acos mows
Feromefe horou gt ten Ne your rown, And iel
All dord I re tre floubcikines Ferdovee n Oh ohneeven sond nele

Thoh Sad ist cand hiok no engs
And I ces lin che elivo owll lids awave sin'tll ome
Ganyou me anvon mit

d mesn ser ne've fre feas wagathafme shak the hounneay evo thime der Allld you'roukthe Seid ounge tring sournelen',
Browo ne"if ked ovean Tha tohujusak ke thig nownaw
You' thome men os olsingon fre owna'st ome taches, cle rget it igotshe,
But houg ed therse da noghhh
Went ging her
Andneve thase wer
Andseaintevere theran hall pyou Thell quesd ad get,
Wharis wol I haprounn tout youbaut oud, fars ty
I's oujuse old thro
Ol mmboh loh oup I keswe toh, sank sud

Liky
Is you is
Whhals fy a you't you've ne tele of gers thesk n't you Bift forugot his simy thhakn' do athow feld toron moucke k

In [ ]:
print( decode(out_2[0].tolist()))

And it was all yellow, look at the starsewr
Bairing iptof
tinghm coef tshantint hro, love dorse thou eldecang wak yeati'm thee mid
Lartlis nigagttieve a'mom
St wheol Anndr
'be I whay rthet
Sooureave ld Thout me yoo th- ona yoru"
Rhind'vey

Wicomoveay me doundnond min't Ooh I the jusler nit dopl wacr baurt bort my
And war umearintw ighe the, dd ig
Igo
Thsayhame na wa,
Fus
I
Icher,
Andin I tg albizeey lvigesif vy tre? Don't de, just flon felovep ofoauck eldd figo now sawas diy you'd ghth ihn's whow,
Twhe
Orplr, yoobreved hink btom fo tu


In [ ]:
print( decode(out_3[0].tolist()))

And it was all yellow, look at the starst,
Ihe me thane canlin call you pally os de whe bat ingond tame ep foll just se bra dald waly I winga ris and ther tin wa tot bed wayt se pour allit's can my beder ght omme whe scaking lin abr, ome thay thes kif yuo sugin I garigiget you tosin ing tar mer be coutrike wigan ave sat ke beay lacain

And thil wang

Dow you n'st your she thing hing, beme,
So mopleveeve st you ta you
I tlo tad bear wthes
But ses ad Don oful dsena mingtth me tigh you't gh aning ay Andse at a ingo a le I fa
Lavound I 'n


In [ ]:
print( decode(out_4[0].tolist()))

And it was all yellow, look at the starsin donoow?
St your per kne
But rshing ow hay I want our oght you, lo ws
No dorund won
Leknad palringet I nikngil osto atence ee? evres os ket halse the
'Caush was daivere've ed to'm abe onking tt mine t's
But no st we tapairso gt, foa mirea ne lab the olf yo
I wasiot you'me ave whe to the lound

I'm derepre oullild I thanid st lak is theesd, soonedo ureamen ngnth ant od'm, ave n"
All bre are
Mommy man gn beat mmaty ohro now ave, mpes me it cads romye the freown, I 'omy,

Ounghs abtu llo re catin


In [ ]:
class SelfAttentionHead(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(embedding_size, head_size, bias = False)
    self.query = nn.Linear(embedding_size, head_size, bias = False)
    self.value = nn.Linear(embedding_size, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
    self.head_size = head_size

  def forward(self, X):
    batch_size, context_length, embedding_size = X.shape
    k = self.key(X)
    q = self.query(X)
    v = self.value(X)

    wei = q @ k.transpose(-2, -1)
    wei = wei / (self.head_size ** 0.5)

    wei = wei.masked_fill(self.tril[:context_length, :context_length]==0, float("-inf"))
    wei = F.softmax(wei, dim = -1)

    out = wei @ v
    return out

In [ ]:
class MultiHeadSelfAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([SelfAttentionHead(head_size) for _ in range(num_heads)])

  def forward(self, X):
    return torch.cat([h(X) for h in self.heads], dim = -1)

In [ ]:
num_heads = 8

In [ ]:
head_size = int(embedding_size / num_heads)

In [ ]:
embedding_size

128

In [ ]:
head_size

16

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.position_embedding_table = nn.Embedding(context_length, embedding_size)
    self.multi_attention_head = MultiHeadSelfAttention(num_heads, head_size)
    self.linear_layer = nn.Linear(embedding_size, vocab_size)     #Final output projection layer, not the feed forward layer

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    position_embedding_output = self.position_embedding_table(torch.arange(T, device = device))
    pos_aware_embedding = embedding_output + position_embedding_output
    multi_attention_head = self.multi_attention_head(pos_aware_embedding)

    logits = self.linear_layer(multi_attention_head) #B, T, vocab_size
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss


  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
      for _ in range(max_new_tokens):
          idx_cond = idx[:, -context_length:]       # truncate to context length
          logits, _ = self(idx_cond)
          logits = logits[:, -1, :]                 # last time step logits

          # 1️⃣ Temperature scaling
          logits = logits / temperature

          # 2️⃣ Top-k filtering (keep k highest logits)
          if top_k is not None:
              values, indices = torch.topk(logits, top_k)
              mask = torch.full_like(logits, float('-inf'))
              mask.scatter_(1, indices, values)
              logits = mask

          # 3️⃣ Top-p (nucleus) filtering
          if top_p is not None:
              sorted_logits, sorted_indices = torch.sort(logits, descending=True)
              probs = F.softmax(sorted_logits, dim=-1)
              cumulative_probs = torch.cumsum(probs, dim=-1)
              cutoff = cumulative_probs > top_p
              cutoff[..., 1:] = cutoff[..., :-1].clone()
              cutoff[..., 0] = False
              sorted_logits[cutoff] = float('-inf')
              logits.scatter_(1, sorted_indices, sorted_logits)

          # 4️⃣ Convert to probabilities and sample next token
          probs = F.softmax(logits, dim=-1)
          idx_next = torch.multinomial(probs, num_samples=1)
          idx = torch.cat((idx, idx_next), dim=1)

      return idx

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(4.3387, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)


In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")


Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 128])

Layer: position_embedding_table
  weight: torch.Size([64, 128])

Layer: multi_attention_head

Layer: multi_attention_head.heads

Layer: multi_attention_head.heads.0

Layer: multi_attention_head.heads.0.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.0.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.0.value
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1

Layer: multi_attention_head.heads.1.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1.value
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2

Layer: multi_attention_head.heads.2.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2.value
  weight: torch.Size([16, 128])

Layer: multi_attent

In [ ]:
for p in m.parameters():
    print(p.shape)

torch.Size([75, 128])
torch.Size([64, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([75, 128])
torch.Size([75])


In [ ]:
for name, param in m.named_parameters():
    print(name, param.shape)


token_embedding_table.weight torch.Size([75, 128])
position_embedding_table.weight torch.Size([64, 128])
multi_attention_head.heads.0.key.weight torch.Size([16, 128])
multi_attention_head.heads.0.query.weight torch.Size([16, 128])
multi_attention_head.heads.0.value.weight torch.Size([16, 128])
multi_attention_head.heads.1.key.weight torch.Size([16, 128])
multi_attention_head.heads.1.query.weight torch.Size([16, 128])
multi_attention_head.heads.1.value.weight torch.Size([16, 128])
multi_attention_head.heads.2.key.weight torch.Size([16, 128])
multi_attention_head.heads.2.query.weight torch.Size([16, 128])
multi_attention_head.heads.2.value.weight torch.Size([16, 128])
multi_attention_head.heads.3.key.weight torch.Size([16, 128])
multi_attention_head.heads.3.query.weight torch.Size([16, 128])
multi_attention_head.heads.3.value.weight torch.Size([16, 128])
multi_attention_head.heads.4.key.weight torch.Size([16, 128])
multi_attention_head.heads.4.query.weight torch.Size([16, 128])
multi_att

In [ ]:

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 76,619


76619

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 10   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with_multiattention_head"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())

Epoch: 1	Train Loss: 1.6074	Eval Loss: 1.7183
1.6073675155639648
Epoch: 2	Train Loss: 1.5435	Eval Loss: 1.9629
1.5435352325439453
Epoch: 3	Train Loss: 1.6015	Eval Loss: 1.7246
1.6014585494995117
Epoch: 4	Train Loss: 1.6205	Eval Loss: 1.5556
1.6204898357391357
Epoch: 5	Train Loss: 1.5502	Eval Loss: 1.4816
1.5501540899276733
Epoch: 6	Train Loss: 1.5937	Eval Loss: 1.5621
1.5937159061431885
Epoch: 7	Train Loss: 1.5659	Eval Loss: 1.5770
1.5659427642822266
Epoch: 8	Train Loss: 1.5889	Eval Loss: 1.5089
1.588856816291809
Epoch: 9	Train Loss: 1.5527	Eval Loss: 1.7359
1.5526577234268188
Epoch: 10	Train Loss: 1.6160	Eval Loss: 1.6642
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_basic_linear_with_multiattention_head/checkpoint_epoch_10.pt
1.6159663200378418
Epoch: 11	Train Loss: 1.5790	Eval Loss: 1.6532
1.5790436267852783


KeyboardInterrupt: 

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_basic_linear_with_multiattention_head/checkpoint_epoch_10.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 10 with train_loss=1.6160


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow, look at the stars"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


And it was all yellow, look at the starst
But was just let we were so paind the dooing sweere now up ath neven
Last scar,
h you wo love
Until, "See ancing time pagone down
It's ple: Edfmilia faime
Is just thout litd been back here cose
That A then stay
For body Jame
'Cause fore tot of tall she gettastell fave left I m storunk arough ave is alm up at you, oh me
But ying my heart take in a know my fear I sed gets

And a the thered

For every eever the the we're oh do
Or hadryning Oh armines out, been warlies they love Sas cutdy of it't parted Wouldn't hud everythind in I neverythings, ah
And I know to hinking
I gueline like spap of wholed through here loe, ooh, my love if yout wear beggire,
To though what it is me head, saured take,
And the rairys your be me, mind,
Trway just her Let lets ma hop where mistake

Look oves in at You new nive you looke on
You stic, hoh, oh

Fight storches so Straid
Uh-uh, oh I peine of the your the the of mom only.


Its your stare to they're I'll with you s

In [ ]:
print( decode(out_2[0].tolist()))

And it was all yellow, look at the stars
Put backs and I know I'm a slet tell be you Yes
Unn"
I'm"
Thine, make what we'd a finting me it when taick, baby
And of thive,
Yeall seet I comentma thered me the must facurning in le awe fight whraen time

Your nevere falloseatly I'm gived one my magevinkin' yourning that cats saitiin the day
Day,
See should the loa
So ast, now tha broke was
Cause smintdmould've grives so in evo,
For soometh is on paake ps bet count the that you'lly coly on,
All it no that thator want juae, made,
Look thind ou


In [ ]:
print( decode(out_3[0].tolist()))

And it was all yellow, look at the stars withat time,
And you the stars and me gived the staking you can the forr light left the sas when you givart gind you'd stop face of my me the lat on thet throut of the pon me and see never that I want you new how at to don't wally ork Cas you don't time

It's never stime thought a but bid down
When fies way break up that
You're stars the flause and the lood gone
My heart to me too and the monds the mothratell go, the stapgot heart when you hould me thing there stonly for the bon, oh, yest it I 


In [ ]:
class SelfAttentionHead(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(embedding_size, head_size, bias = False)
    self.query = nn.Linear(embedding_size, head_size, bias = False)
    self.value = nn.Linear(embedding_size, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
    self.head_size = head_size

  def forward(self, X):
    batch_size, context_length, embedding_size = X.shape
    k = self.key(X)
    q = self.query(X)
    v = self.value(X)

    wei = q @ k.transpose(-2, -1)
    wei = wei / (self.head_size ** 0.5)

    wei = wei.masked_fill(self.tril[:context_length, :context_length]==0, float("-inf"))
    wei = F.softmax(wei, dim = -1)

    out = wei @ v
    return out

In [ ]:
class MultiHeadSelfAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([SelfAttentionHead(head_size) for _ in range(num_heads)])


  def forward(self, X):
    return torch.cat([h(X) for h in self.heads], dim = -1)

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, embedding_size ):
    super().__init__()
    self.fforward = nn.Sequential(
    nn.Linear(embedding_size, 4 * embedding_size),
    nn.ReLU(),
    nn.Linear(4 * embedding_size, embedding_size))

  def forward(self, X):
    return(self.fforward(X))


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.position_embedding_table = nn.Embedding(context_length, embedding_size)
    self.multi_attention_head = MultiHeadSelfAttention(num_heads, head_size)
    self.feed_forward_layer = FeedForward(embedding_size)
    self.linear_layer = nn.Linear(embedding_size, vocab_size)     #Final output projection layer, not the feed forward layer

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    position_embedding_output = self.position_embedding_table(torch.arange(T, device = device))
    pos_aware_embedding = embedding_output + position_embedding_output
    multi_attention_head = self.multi_attention_head(pos_aware_embedding)
    ff_output = self.feed_forward_layer(multi_attention_head)

    logits = self.linear_layer(ff_output)
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss


  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]       # truncate to context length
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :]                 # last time step logits

        # 1️⃣ Temperature scaling
        logits = logits / temperature

        # 2️⃣ Top-k filtering (keep k highest logits)
        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            mask = torch.full_like(logits, float('-inf'))
            mask.scatter_(1, indices, values)
            logits = mask

        # 3️⃣ Top-p (nucleus) filtering
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(probs, dim=-1)
            cutoff = cumulative_probs > top_p
            cutoff[..., 1:] = cutoff[..., :-1].clone()
            cutoff[..., 0] = False
            sorted_logits[cutoff] = float('-inf')
            logits.scatter_(1, sorted_indices, sorted_logits)

        # 4️⃣ Convert to probabilities and sample next token
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(4.3446, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)


In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")


Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 128])

Layer: position_embedding_table
  weight: torch.Size([64, 128])

Layer: multi_attention_head

Layer: multi_attention_head.heads

Layer: multi_attention_head.heads.0

Layer: multi_attention_head.heads.0.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.0.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.0.value
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1

Layer: multi_attention_head.heads.1.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.1.value
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2

Layer: multi_attention_head.heads.2.key
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2.query
  weight: torch.Size([16, 128])

Layer: multi_attention_head.heads.2.value
  weight: torch.Size([16, 128])

Layer: multi_attent

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 208,331


208331

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 2   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())

Epoch: 1	Train Loss: 1.1886	Eval Loss: 1.4293
1.1886061429977417
Epoch: 2	Train Loss: 1.2373	Eval Loss: 1.2110
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward/checkpoint_epoch_2.pt
1.2372545003890991
Epoch: 3	Train Loss: 1.0873	Eval Loss: 1.3902
1.0873138904571533
Epoch: 4	Train Loss: 0.9583	Eval Loss: 1.4914
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward/checkpoint_epoch_4.pt
0.9582810401916504
Epoch: 5	Train Loss: 1.1140	Eval Loss: 1.2572
1.1140414476394653
Epoch: 6	Train Loss: 1.2751	Eval Loss: 1.7472
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward/checkpoint_epoch_6.pt
1.2750799655914307
Epoch: 7	Train Loss: 1.0703	Eval Loss: 1.4500
1.0702933073043823
Epoch: 8	Train Loss: 1.0167	Eval Loss: 1.8347
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward/checkpoint_epoch_8.pt
1.0167018175125122
Epoc

KeyboardInterrupt: 

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_with_multiattention_head_and_fforward/checkpoint_epoch_2.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 2 with train_loss=1.2373


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow, look at the stars"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


And it was all yellow, look at the stars

But was just late supermas a long that your name with my Aymarring goes
There are you won't know on, it's always cause always before
Friends time times, this life touch like it babe?

Tos thesebill be loving can wrapon
And manting to with these sheets up to say side pull peach thornnange things and a mad knew it's all I knowing unbe
With ah ah
And your shit
Just on gets
And I will trying out
He rep every time is toma 's all so gwing in love much, could be comerap

Welly love me, cury going it pain

Choresn't not livide it goes single vow
You and me right
Ah

And aga
Reems
And give
Cropuive a lie tight. oh'Cause I don\ut wanna know at way forget
Ooh
Maybe beautiful my castle breeze for a locked, my face
Saying to meet
My heart is and her Light
Make it's where mistake

Love me shine

Can and now,
that I'm wanna fade it all from get
Sleep made Green
Saying as the fits in the castle of
Wond I was him with a
Are your name
Your love it off anti-hoh-a

In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("Romeo take me somewhere I can be alone"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


Romeo take me somewhere I can be alone
I wish I can spells
Josebory dancing a keep home. They'll sing for smoke, so pathing my roombodmoment of the next to meet there
Sitting home, that I heard the path feeling sange the dark
Keep you of your heal that you get, shake it love,
She same yard friend a Feelse the day singeroud

Cropul ruborn with that midnight, sch, ah-ho, oo

faith insish thmustarts, they're anymorle of you don't your ropped up, up, up when you're anymore thand
You say and begin,
And I thinkid
I'm not fifteen
If I maybe thesing in the mountains
If the still haue...

Your eyes

I guess a waterfation
Pa just a fricks on a tried to body's when you didn't have you mitherough
But what you everry stordance me
But I've known
'Cause your singing must that I gave youer night in your body,
And get that your gardest forever in different mystear
Sucars,
From the silence, love
Take work
Friends are treak on the handom the hundred up on up eath fight
I'm awas always before If it son't 

In [ ]:
print( decode(out_2[0].tolist()))

Romeo take me somewhere I can be alone
All you'll the can kid,
Give alright only
Ank that's moving me halfwat it to danca dold my love, long more like you
Every day body chase. of Babe, baby, Me does,
And I might now,
I'm dancean
, I'm stay that kif you sunround criging,
I didn't spen
Barcelona! Baby, I pashed a costand see, go
Sing

And this wanging to get is nothing
And it feels, baby, So I'll been my chemica you

I love down hanted marress trust s off
Just a making

I'm in love can maring again
Your loving ach and hang, let insin


In [ ]:
print( decode(out_3[0].tolist()))

Romeo take me somewhere I can be alonely know
I'm not you are never make my own
Take a long was good
When you
When you're thurts down the same all of town
Let me see?
I really into the mome
And I love you tonighter to find of kind of me
I tried to when we got to let my lover land head to soldie the sunrish,
'mm all at only being you
I'm in love with you were single would you I'm so dead up a perfect dance doncean

In the right I think your back
Now still rose right
I pushed into that you to breaking I won't wanna know heart in the b


In [ ]:
train_batch_size = 128
context_length = 64

embedding_size = 64

num_heads = 4

head_size = int(embedding_size / num_heads)
head_size

16

In [ ]:
class SelfAttentionHead(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(embedding_size, head_size, bias = False)
    self.query = nn.Linear(embedding_size, head_size, bias = False)
    self.value = nn.Linear(embedding_size, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
    self.head_size = head_size

  def forward(self, X):
    batch_size, context_length, embedding_size = X.shape
    k = self.key(X)
    q = self.query(X)
    v = self.value(X)

    wei = q @ k.transpose(-2, -1)
    wei = wei / (self.head_size ** 0.5)

    wei = wei.masked_fill(self.tril[:context_length, :context_length]==0, float("-inf"))
    wei = F.softmax(wei, dim = -1)

    out = wei @ v
    return out

In [ ]:
class MultiHeadSelfAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([SelfAttentionHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(embedding_size, embedding_size)
    self.dropout = nn.Dropout(0.1)


  def forward(self, X):
    output = torch.cat([h(X) for h in self.heads], dim = -1)
    output = self.projection(output)
    output = self.dropout(output)
    return output

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, embedding_size ):
    super().__init__()
    self.fforward = nn.Sequential(
    nn.Linear(embedding_size, 2 * embedding_size),
    nn.GELU(),
    nn.Linear(2 * embedding_size, embedding_size), nn.Dropout(0.1) )

  def forward(self, X):
    return(self.fforward(X))


In [ ]:
class DecoderBlock(nn.Module):

  def __init__(self, embedding_size):
    super().__init__()
    self.multi_attention_head = MultiHeadSelfAttention(num_heads, head_size)
    self.feed_forward_layer = FeedForward(embedding_size)
    self.layer_norm1 = nn.LayerNorm(embedding_size)
    self.layer_norm2 = nn.LayerNorm(embedding_size)


  def forward(self, X):
    X = X + self.multi_attention_head(self.layer_norm1(X))
    out = X + self.feed_forward_layer(self.layer_norm2(X))
    return out

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.position_embedding_table = nn.Embedding(context_length, embedding_size)
    self.dropout = nn.Dropout(0.1)
    self.decoder = nn.Sequential(DecoderBlock(embedding_size),
                                 DecoderBlock(embedding_size),
                                 DecoderBlock(embedding_size),
                                 nn.LayerNorm(embedding_size))
    self.linear_layer = nn.Linear(embedding_size, vocab_size)     #Final output projection layer, not the feed forward layer

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    position_embedding_output = self.position_embedding_table(torch.arange(T, device = device))
    pos_aware_embedding = embedding_output + position_embedding_output
    pos_aware_embedding = self.dropout(pos_aware_embedding)
    output = self.decoder(pos_aware_embedding)

    logits = self.linear_layer(output)
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss


  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]       # truncate to context length
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :]                 # last time step logits

        # 1️⃣ Temperature scaling
        logits = logits / temperature

        # 2️⃣ Top-k filtering (keep k highest logits)
        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            mask = torch.full_like(logits, float('-inf'))
            mask.scatter_(1, indices, values)
            logits = mask

        # 3️⃣ Top-p (nucleus) filtering
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(probs, dim=-1)
            cutoff = cumulative_probs > top_p
            cutoff[..., 1:] = cutoff[..., :-1].clone()
            cutoff[..., 0] = False
            sorted_logits[cutoff] = float('-inf')
            logits.scatter_(1, sorted_indices, sorted_logits)

        # 4️⃣ Convert to probabilities and sample next token
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(4.5072, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
for name, module in m.named_modules():
    print(f"\nLayer: {name}")
    for param_name, param in module.named_parameters(recurse=False):
        print(f"  {param_name}: {param.shape}")


Layer: 

Layer: token_embedding_table
  weight: torch.Size([75, 64])

Layer: position_embedding_table
  weight: torch.Size([64, 64])

Layer: dropout

Layer: decoder

Layer: decoder.0

Layer: decoder.0.multi_attention_head

Layer: decoder.0.multi_attention_head.heads

Layer: decoder.0.multi_attention_head.heads.0

Layer: decoder.0.multi_attention_head.heads.0.key
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.0.query
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.0.value
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.1

Layer: decoder.0.multi_attention_head.heads.1.key
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.1.query
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.1.value
  weight: torch.Size([16, 64])

Layer: decoder.0.multi_attention_head.heads.2

Layer: decoder.0.multi_attention_head.heads.2.key
  weight: torch.Size([16, 64])

Layer: d

In [ ]:
for name, param in m.named_parameters():
    print(name, param.shape)

token_embedding_table.weight torch.Size([75, 64])
position_embedding_table.weight torch.Size([64, 64])
decoder.0.multi_attention_head.heads.0.key.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.0.query.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.0.value.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.1.key.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.1.query.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.1.value.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.2.key.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.2.query.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.2.value.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.3.key.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.3.query.weight torch.Size([16, 64])
decoder.0.multi_attention_head.heads.3.value.weight torch.Size([16, 64])
decoder.0.multi_attention_hea

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 113,739


113739

In [ ]:
# import torch
# import os

# # assume: m, optimizer, train_loader, eval_loader already defined
# epochs = 50
# steps_per_epoch = len(train_data) // train_batch_size
# save_every = 1   # save checkpoint every 10 epochs
# checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation"
# os.makedirs(checkpoint_dir, exist_ok=True)

# for ep in range(epochs):
#     m.train()
#     for step in range(steps_per_epoch):
#         Xb, Yb = train_loader.get_batch()
#         logits, loss = m(Xb, Yb)
#         optimizer.zero_grad(set_to_none=True)
#         loss.backward()
#         optimizer.step()

#     # evaluate at the end of each epoch
#     m.eval()
#     with torch.no_grad():
#         xvb, yvb = eval_loader.get_batch()
#         _, e_loss = m(xvb, yvb)
#     print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

#     # save checkpoint every N epochs
#     if (ep + 1) % save_every == 0:
#         checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
#         torch.save({
#             "epoch": ep + 1,
#             "model_state": m.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "train_loss": loss.item(),
#             "eval_loss": e_loss.item(),
#         }, checkpoint_path)
#         print(f"💾 Saved checkpoint: {checkpoint_path}")

#     m.train()
#     print(loss.item())


In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 2   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())


Epoch: 1	Train Loss: 1.4273	Eval Loss: 1.4399
1.4272602796554565
Epoch: 2	Train Loss: 1.3186	Eval Loss: 1.3426
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version/checkpoint_epoch_2.pt
1.3185973167419434
Epoch: 3	Train Loss: 1.3548	Eval Loss: 1.4655
1.3548192977905273
Epoch: 4	Train Loss: 1.3809	Eval Loss: 1.3883
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version/checkpoint_epoch_4.pt
1.3809442520141602
Epoch: 5	Train Loss: 1.3663	Eval Loss: 1.3975
1.3662538528442383
Epoch: 6	Train Loss: 1.3483	Eval Loss: 1.3823
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version/checkpoint_epoch_6.pt
1.3483439683914185
Epoch: 7	Train Loss: 1.3550	Eval Loss: 1.1925
1.354978322982788
Epoch: 8	Train Loss: 1.2849	Eval Loss: 1.0076
💾 Saved checkpoint: /content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version/checkpoint

KeyboardInterrupt: 

In [ ]:
# checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation/checkpoint_epoch_2.pt"  # or whichever you saved

# checkpoint = torch.load(checkpoint_path, map_location=device)

# m.load_state_dict(checkpoint["model_state"])
# optimizer.load_state_dict(checkpoint["optimizer_state"])
# start_epoch = checkpoint["epoch"]

# print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 2 with train_loss=1.0419


In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/checkpoints_with_full_decoder_implementation_small_version/checkpoint_epoch_8.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


✅ Resumed from epoch 8 with train_loss=1.2849


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow, look at the stars"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


And it was all yellow, look at the stars warm bleaning,
To you again at qoo I whiced a lot once me-I does in love a lot where ighre,
And back the sun drien is a night,
Broke,
The lier what I
Da wait-of your hor oCed,
Bleight,
It there was cruest sen to get me
Go togething again't heart need No les
Everythi can was a sided the jusure cry stand
And I don't wy?
For oitning of star
It all said ask up you there turn the wenery In chabs
They get Home away of me everry time
Ain't could roa-le both confused
Oh Oh I don't talk me, that's that I losed have been rain X
Oh I, you so've been, wait away, I don't then want you today I\um from looked
He right one a with ortmost a rips you,
The wway it countrong linest Lalw-lood,
Teningaise to the side
Ligway, hand you were, don't save with a lover night
Sword whated I sing you kick Of no else tony awaying founstawero
Don't you do do


Oh, mus
I can't put if you would nevon to do no be come down your bean down I don't keep front

And wantbent to so you

In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("Romeo take me somewhere I can be alone"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


Romeo take me somewhere I can be alone
And I know apon's about ee
And then I was mucan if how can it said line you got the world eyday on you look theo fells
You don't fall lius somethinxing then, they always for keep wined fall,
Your own eedy road thatrolen

You dear in a kend in But you no?
Anot place is a moment come ,
You should seat you of the days somehow
Shing better when perfinged holow, though oh oh oh oh oh oh ooh
Ooooooh ooh oh ooh oohhhhh Oooooooooo oven hey life living lound

You heard I\uch
Brount no on and mell I've shile but you ward at I could sell tighting my need
Stup bight throught, earbled divi down I'll find a pain
Slowf, the home happed a drigger is a runoor come home
Someone all of the rose entmreta, breaked


Far it right the way you said go 'to the growing up, person your moray good that Feeling
And you see a people
And caskma to the lung to can where we go no well be we watchange and I think they say know that you need me
Aple if you below this is is the grow

In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("Love"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


Love go merever With me,
I guess at tuy perfect beaut to back to falls,
For
Got to lock just keeping at penty,
Judie with out up digh of a truth soos Blear,
It deving up therasering had to say one blefing on the world
And you loud, we don't Need in to, how I've try mord one you to see that out aema

You have ran ome on that haze
No Yo oh
I fire you too

Uh, I look The falls it a changer house I close a side get out of stillying on my mind of lot losts you"

Dou as make fig anyïurs just what if I don't it feels
If all in my honey more handswaying 'bout it night
NA come I on
And I wanted to ask the rivin sof it her daylizare body adone

It's a rchose and 'cause I showed sparted July bort the soard wise,
Can shels the spearks us beforgiat,
This and 'round the raight of love,
And her clords out toge,
It's ruckligh thede noom in though the night
Name on eplath you can lost on Days is doing anything you want to get off"

And I'm maybe?

Are I will be going world, sking on the bitches be to s

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

# Use GPT-2 tokenizer (same as OpenAI models)
tokenizer = tiktoken.get_encoding("gpt2")

# Vocabulary size from tokenizer
vocab_size = tokenizer.n_vocab
print("Vocab size:", vocab_size)  # typically 50257


Vocab size: 50257


In [ ]:
len(text)

903785

In [ ]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long, device=device)
print(f"Total tokens: {len(tokens)}")

Total tokens: 254491


In [ ]:
# Optional sanity check
print("Total tokens:", len(tokens))
print("First 50 tokens:", tokens[:50])


Total tokens: 254491
First 50 tokens: tensor([10044,   620,  1769,   220,   198,   198,    33,  1952, 27141,   588,
        14966,  1439,   262,   356,  2121,   329,   198,  3237,   883,  4113,
          356,  1053,  7334,  1439,   286,   514,   389,  1760,   329,   198,
          198,  1135,  2107,   287,   257,  4950,   995,  9425,   356,   466,
           11, 10194,   356,   466,   775,  2107,   287,   257,  4950,   995],
       device='cuda:0')


In [ ]:
# --- Split train/validation ---
split = int(0.9 * len(tokens))
train_tokens = tokens[:split]
val_tokens = tokens[split:]

In [ ]:
import torch

class GPTDataLoader:
    def __init__(self, tokens, batch_size, context_length):
        self.tokens = tokens
        self.batch_size = batch_size
        self.context_length = context_length
        self.current_position = 0

    def get_batch(self) -> tuple[torch.Tensor, torch.Tensor]:
        b, c = self.batch_size, self.context_length
        n = len(self.tokens)

        start_pos = self.current_position
        end_pos = start_pos + b * c + 1
        d = self.tokens[start_pos:end_pos]

        # wrap-around if needed
        if end_pos > n:
          overflow = end_pos - n
          d = torch.cat([self.tokens[start_pos:], self.tokens[:overflow]])

        x = d[:-1].view(b, c)
        y = d[1:].view(b, c)

        self.current_position = (self.current_position + b * c) % n

        return x, y

In [ ]:
# --- DataLoader initialization ---
train_loader = GPTDataLoader(train_tokens, batch_size=128, context_length=64)
eval_loader = GPTDataLoader(val_tokens, batch_size=128, context_length=64)

# --- Example batch ---
Xb, Yb = train_loader.get_batch()
print("Batch shapes:", Xb.shape, Yb.shape)

Batch shapes: torch.Size([128, 64]) torch.Size([128, 64])


In [ ]:
class SelfAttentionHead(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(embedding_size, head_size, bias = False)
    self.query = nn.Linear(embedding_size, head_size, bias = False)
    self.value = nn.Linear(embedding_size, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))
    self.head_size = head_size

  def forward(self, X):
    batch_size, context_length, embedding_size = X.shape
    k = self.key(X)
    q = self.query(X)
    v = self.value(X)

    wei = q @ k.transpose(-2, -1)
    wei = wei / (self.head_size ** 0.5)

    wei = wei.masked_fill(self.tril[:context_length, :context_length]==0, float("-inf"))
    wei = F.softmax(wei, dim = -1)

    out = wei @ v
    return out

In [ ]:
class MultiHeadSelfAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([SelfAttentionHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(embedding_size, embedding_size)
    self.dropout = nn.Dropout(0.1)


  def forward(self, X):
    output = torch.cat([h(X) for h in self.heads], dim = -1)
    output = self.projection(output)
    output = self.dropout(output)
    return output

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, embedding_size ):
    super().__init__()
    self.fforward = nn.Sequential(
    nn.Linear(embedding_size, 2 * embedding_size),
    nn.GELU(),
    nn.Linear(2 * embedding_size, embedding_size), nn.Dropout(0.1) )

  def forward(self, X):
    return(self.fforward(X))


In [ ]:
class DecoderBlock(nn.Module):

  def __init__(self, embedding_size):
    super().__init__()
    self.multi_attention_head = MultiHeadSelfAttention(num_heads, head_size)
    self.feed_forward_layer = FeedForward(embedding_size)
    self.layer_norm1 = nn.LayerNorm(embedding_size)
    self.layer_norm2 = nn.LayerNorm(embedding_size)


  def forward(self, X):
    X = X + self.multi_attention_head(self.layer_norm1(X))
    out = X + self.feed_forward_layer(self.layer_norm2(X))
    return out

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1227)

class GPT(nn.Module):

  def __init__(self, vocab_size, embedding_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
    self.position_embedding_table = nn.Embedding(context_length, embedding_size)
    self.dropout = nn.Dropout(0.1)
    self.decoder = nn.Sequential(DecoderBlock(embedding_size),
                                 DecoderBlock(embedding_size),
                                 DecoderBlock(embedding_size),
                                 nn.LayerNorm(embedding_size))
    self.linear_layer = nn.Linear(embedding_size, vocab_size)     #Final output projection layer, not the feed forward layer

  def forward(self, idx, targets=None):

    B, T = idx.shape
    embedding_output = self.token_embedding_table(idx)  #B, T, embedding_size
    position_embedding_output = self.position_embedding_table(torch.arange(T, device = device))
    pos_aware_embedding = embedding_output + position_embedding_output
    pos_aware_embedding = self.dropout(pos_aware_embedding)
    output = self.decoder(pos_aware_embedding)

    logits = self.linear_layer(output)
    loss = None
    if targets is not None:
        batch_size, sequence_length, d_model = logits.shape
        logits = logits.view(batch_size*sequence_length, d_model)
        targets = targets.view(batch_size*sequence_length)
        loss = F.cross_entropy(logits, targets)
    return logits, loss


  def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]       # truncate to context length
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :]                 # last time step logits

        # 1️⃣ Temperature scaling
        logits = logits / temperature

        # 2️⃣ Top-k filtering (keep k highest logits)
        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            mask = torch.full_like(logits, float('-inf'))
            mask.scatter_(1, indices, values)
            logits = mask

        # 3️⃣ Top-p (nucleus) filtering
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(probs, dim=-1)
            cutoff = cumulative_probs > top_p
            cutoff[..., 1:] = cutoff[..., :-1].clone()
            cutoff[..., 0] = False
            sorted_logits[cutoff] = float('-inf')
            logits.scatter_(1, sorted_indices, sorted_logits)

        # 4️⃣ Convert to probabilities and sample next token
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
m = GPT(vocab_size, embedding_size).to(device)
logits, loss = m(Xb, Yb)
print(loss)

tensor(10.9582, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
optimizer =  torch.optim.AdamW(m.parameters(), lr = 1e-3)

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total:,}")
    return total

count_parameters(m)

Total trainable parameters: 6,587,217


6587217

In [ ]:
vocab_size

50257

In [ ]:
import torch
import os

# assume: m, optimizer, train_loader, eval_loader already defined
epochs = 50
steps_per_epoch = len(train_data) // train_batch_size
save_every = 1   # save checkpoint every 10 epochs
checkpoint_dir = "/content/drive/MyDrive/GPT/full_tiktoken"
os.makedirs(checkpoint_dir, exist_ok=True)

for ep in range(epochs):
    m.train()
    for step in range(steps_per_epoch):
        Xb, Yb = train_loader.get_batch()
        logits, loss = m(Xb, Yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # evaluate at the end of each epoch
    m.eval()
    with torch.no_grad():
        xvb, yvb = eval_loader.get_batch()
        _, e_loss = m(xvb, yvb)
    print(f"Epoch: {ep+1}\tTrain Loss: {loss:.4f}\tEval Loss: {e_loss:.4f}")

    # save checkpoint every N epochs
    if (ep + 1) % save_every == 0:
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{ep+1}.pt"
        torch.save({
            "epoch": ep + 1,
            "model_state": m.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "train_loss": loss.item(),
            "eval_loss": e_loss.item(),
        }, checkpoint_path)
        print(f"💾 Saved checkpoint: {checkpoint_path}")

    m.train()
    print(loss.item())


Epoch: 1	Train Loss: 3.7047	Eval Loss: 3.3242
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_1.pt
3.7047231197357178
Epoch: 2	Train Loss: 2.5490	Eval Loss: 2.9295
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_2.pt
2.549009323120117
Epoch: 3	Train Loss: 2.0308	Eval Loss: 3.4910
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_3.pt
2.0308425426483154
Epoch: 4	Train Loss: 1.7368	Eval Loss: 2.8823
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_4.pt
1.7368048429489136
Epoch: 5	Train Loss: 1.4446	Eval Loss: 2.3232
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_5.pt
1.4446327686309814
Epoch: 6	Train Loss: 1.3534	Eval Loss: 3.0135
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_6.pt
1.3533612489700317
Epoch: 7	Train Loss: 1.3181	Eval Loss: 3.3758
💾 Saved checkpoint: /content/drive/MyDrive/GPT/full_tiktoken/checkp

In [ ]:
checkpoint_path = "/content/drive/MyDrive/GPT/full_tiktoken/checkpoint_epoch_8.pt"  # or whichever you saved

checkpoint = torch.load(checkpoint_path, map_location=device)

m.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

print(f"✅ Resumed from epoch {start_epoch} with train_loss={checkpoint['train_loss']:.4f}")


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("And it was all yellow, look at the stars"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))


In [ ]:
with torch.no_grad():
    context = torch.tensor(encode("love"), dtype=torch.long, device=device).unsqueeze(0)

    # deterministic: low randomness
    out_1 = m.generate(context, max_new_tokens=1000, temperature=1)

    # creative: more diverse tokens
    out_2 = m.generate(context, max_new_tokens=500, temperature=1.2, top_k=25)

    # GPT-style “nucleus” sampling
    out_3 = m.generate(context, max_new_tokens=500, temperature=0.8, top_p=1)

    # GPT-style “nucleus” sampling
    out_4 = m.generate(context, max_new_tokens=500, temperature=1.0, top_k=25)

    print( decode(out_1[0].tolist()))
